# Hooks as event listeners

A service tells amoorion that something happened: `POST /hooks/<name>`. Every hook
verifies its caller first, and one without its secret refuses everything.

In [1]:
import os

os.environ["AMOORION_SECRET_HOOKS_GENERIC_TOKEN"] = "made-up"

os.environ["AMOORION_SECRET_HOOKS_SENTRY_SECRET"] = "made-up"

os.environ["AMOORION_SECRET_HOOKS_PLANE_SECRET"] = "made-up"

In [2]:
import hashlib, hmac, json

from fastapi.testclient import TestClient

from amoorion import Amoorion

from amoorion.hooks import HOOKS

amo = Amoorion()

client = TestClient(amo.api())

HOOKS.names()

['discord', 'generic', 'github', 'gitlab', 'plane', 'sentry']

The simplest takes a shared token:

In [3]:
client.post(
    "/hooks/generic",
    json={"text": "deploy finished", "author": "ci"},
    headers={"X-Amoorion-Token": "made-up"},
).json()["text"]

'deploy finished'

Others sign the body. A wrong signature is refused:

In [4]:
def sign(payload):

    body = json.dumps(payload).encode()

    return body, hmac.new(b"made-up", body, hashlib.sha256).hexdigest()


sentry_body, sentry_signature = sign(
    {
        "action": "created",
        "data": {
            "issue": {
                "title": "Boom",
                "permalink": "https://example.org/1",
            }
        },
    }
)


print(
    client.post(
        "/hooks/sentry",
        content=sentry_body,
        headers={
            "Sentry-Hook-Signature": "wrong",
            "Sentry-Hook-Resource": "issue",
        },
    ).status_code
)

print(
    client.post(
        "/hooks/sentry",
        content=sentry_body,
        headers={
            "Sentry-Hook-Signature": sentry_signature,
            "Sentry-Hook-Resource": "issue",
        },
    ).json()["text"]
)

403
sentry: issue created: Boom


In [5]:
plane_body, plane_signature = sign(
    {
        "event": "workitem.created",
        "event_id": "e1",
        "data": {"name": "Fix the door"},
    }
)

client.post(
    "/hooks/plane",
    content=plane_body,
    headers={
        "X-Plane-Signature": plane_signature,
        "X-Plane-Event": "workitem.created",
    },
).json()["text"]

'plane: workitem.created: Fix the door'

All of it lands in the inbox, where rules act on it:

In [6]:
[message["text"] for message in amo.inbox()]

['plane: workitem.created: Fix the door',
 'sentry: issue created: Boom',
 'deploy finished']